_Connecting to Python 3.13.1..._

In [ ]:
from pyexpat import model

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
from pathlib import Path

#-----OVERALL GOAL-------
#How to predict the peak of the epidemic?
    #What do we need to be able to predict the future?
    #Step 1: Figure out how to solve the SEIR model
    #Step 2: Fit a model to our data by picking good parameters
    #Step 3: Extrapolate past the data that we have to see when I(t) peaks in the model

# Load the data
# find the folder where the script is located
HERE = Path(__file__).parent

# build path to the csv
csv_path = HERE / "mystery_virus_daily_active_counts_RELEASE#2.csv"

data = pd.read_csv(csv_path, parse_dates=['date'])

#-----------STEP 1---------------
#Pseudocode for Euler’s Method Approximation of SEIR
    #INPUTS: beta, sigma, gamma, S0, E0, I0, R0, timepoints, N
    #• Initialize S, E, I, and R as empty arrays or lists
    #• Set first item in each list equal to initial values S0, E0, I0, R0
    ###• For each timepoint in timepoints:
    #• Calculate the four derivatives at timepoint
    #• Calculate S, E, I, and R at timepoint + 1 using Euler’s method
    #• Return S, E, I, and R

#euler's method to approximate SEIR using function
# INPUTS: beta, sigma, gamma, S0, E0, I0, R0, timepoints, N
def eulerapprox_SEIR(beta, sigma, gamma, S0, E0, I0, R0, timepoints, N):

    #Initialize S, E, I, R as empty arrays
    S = np.zeros(len(timepoints)) #np.zeros creates an array of zeros with the same length as timepoints (days)
    E = np.zeros(len(timepoints))
    I = np.zeros(len(timepoints))
    R = np.zeros(len(timepoints))

    #Set first item in each list equal to initial values  S0, E0, I0, R0
    S[0] = S0
    E[0] = E0
    I[0] = I0
    R[0] = R0

    #for each timepoint in timepoints:
    for t in range(len(timepoints) - 1): #len(timepoints) - 1 here because we are calculating the next timepoint values / the next day values, so we stop one day before the end 

    #Calculate four derivatives at timepoint
        
        # Change in susceptible population S
        dS = -beta * ((S[t] * I[t]) / N)
        
        # Change in exposed population E
        dE = (beta * ((S[t] * I[t]) / N)) - (sigma * E[t])
        
        # Change in infected population I
        dI = (sigma * E[t]) - (gamma * I[t])
        
        # Change in recovered population R
        dR = gamma * I[t]

    #Calculate S, E, I, and R at timepoint + 1 using Euler’s method ---> Use Euler's Method to calculate next day's values
    # euler's method: y(i+1) = y(i) + (dy/dt)*h ---> next value = current value + (derivative * time step) ---> h here is 1 day so we *1

        S[t+1] = S[t] + dS #calculates next day's susceptible population
        E[t+1] = E[t] + dE #calculates next day's exposed population
        I[t+1] = I[t] + dI #calculates next day's infected population
        R[t+1] = R[t] + dR #calculates next day's recovered population

    #Return S, E, I, and R
    return S, E, I, R
#----------------------------------------------------


#-----------STEP 2---------------
#fitting the SEIR model by picking good parameters

#Our goal: get S(t), E(t), I(t), and R(t) using Euler’s method
    #• These solutions will depend on our parameters, beta, sigma, and gamma
    #• We can choose different values of the parameters to try to match our actual dataset
    #• In other words, we will combine Euler’s method with optimization of SSE to pick best beta, sigma, and gamma for our dataset

#Grid search optimization pseudocode
    #INPUTS: timepoints, N, S0, E0, I0, R0, data
    #• Initialize a range for beta, sigma, and gamma
    #• Initialize an empty array of SSE
    #• Make arrays of values given each range for each parameter
    #• For b in beta
        #• For s in sigma
            #• For g in gamma
                #• Use the Euler method function you developed to calculate S, E, I, and R given those parameters
                #• Calculate the SSE given the model results and the data and append this to the SSE array
    #• Determine parameters corresponding to lowest SSE
    #• Return best_beta, best_sigma, and best_gamma and corresponding SSE

#optimization function
#INPUTS: timepoints, N, S0, E0, I0, R0, data
def optimize_SEIR(timepoints, N, S0, E0, I0, R0, data):

    #• Initialize a range for beta, sigma, and gamma
    gamma_range = np.linspace(0.111,0.2,20) # gamma = recovery rate = 1/symptomatic period, so range of 1/9 to 1/5
    sigma_range = np.linspace(0.056,0.083,30) #sigma = incubation rate = 1/incubation period, so range of 1/18 to 1/12
    r0 = 2.0930 #r0 found in our analysis
    beta_range = np.linspace(r0*gamma_range[0], r0*gamma_range[-1], 20) #beta = r0 * gamma, so we can calculate the range of beta using the range of gamma and r0

    #initialize an empty array of SSE
    SSE_array = []
    parameters = [] #this is list of paramters (beta, sigma, gamma) 

    #make arrays of values given each range for each parameter
    for b in beta_range:
        for s in sigma_range:
            for g in gamma_range:
                #Use euler method function you developed to calculate S, E, I, and R given those parameters
                S,E,I,R = eulerapprox_SEIR(b, s, g, S0, E0, I0, R0, timepoints, N)
                #calculate the SSE given the model results and the data and append this to the SSE array
                SSE = np.sum((I - data['active reported daily cases'])**2) #sse = sum of (I model - I data)^2
                SSE_array.append(SSE)
                parameters.append((b, s, g)) #this adds the parameters to the list
    
    #Determine parameters corresponding to lowest SSE
    corres_SSE = SSE_array.index(min(SSE_array)) #this finds the index of the minimum SSE in the SSE array, which corresponds to the best parameters
    best_beta, best_sigma, best_gamma = parameters[corres_SSE] #this finds the best parameters using the index of the minimum SSE

    #return best_beta, best_sigma, best_gamma and correspondsing SSE
    return best_beta, best_sigma, best_gamma, corres_SSE

#-----------------------------------------------------


#-----------STEP 3---------------
#predicting the peak 

#INPUTS: best_beta, best_sigma, best_gamma
    #• Use the Euler method to run the model out many more days until the data peaks
    #• How high is the peak? Is that a reasonable value?
    #• What day will the peak occur?

def predict_peak(best_beta, best_sigma, best_gamma):
   
#-----------STEP 3---------------
#predicting the peak 
# 
#INPUTS: best_beta, best_sigma, best_gamma
# • Use Euler's method to run the model many more days
# • Find when I(t) reaches its maximum
# • Report peak size and peak date

    def predict_peak(best_beta, best_sigma, best_gamma):    
    # extend simulation beyond observed data    
        extra_days = 200      
        total_days = len(data) + extra_days    
        time_ext = np.arange(total_days)    
    
    # run the SEIR model with best-fit parameters    
    S, E, I, R = eulerapprox_SEIR(best_beta, best_sigma, best_gamma,                                  S0, E0, I0, R0, time_ext, N)    
    
    # find peak of I(t)    
    peak_idx = np.argmax(I)    
    peak_value = I[peak_idx]    
    
    # find the date corresponding to the peak    
    start_date = data['date'].iloc[0]    
    peak_date = start_date + pd.Timedelta(days=int(peak_idx))    
    
    # plot   
    plt.figure(figsize=(10,4))    
    plt.plot(data['date'], data['active reported daily cases'], 'ro', label='data')    
    full_dates = pd.date_range(start=start_date, periods=total_days)    
    plt.plot(full_dates, I, label='model I(t)')    
    plt.axvline(peak_date, color='k', linestyle='--')    
    plt.title("Predicted Peak of Infection")    
    plt.xlabel("Date")    
    plt.ylabel("I(t)")    
    plt.legend()    
    plt.tight_layout()    
    plt.show()    
    print("Predicted peak infected:", peak_value)    
    print("Predicted peak date:", peak_date.date())    
    return peak_value, peak_date





#Next steps in your code:
    #Implement Euler’s method for SEIR modeling
    #• Plot Euler’s method solutions for I(t) and compare to your data
    #• Guess beta, sigma, and gamma and calculate SSE